In [3]:
import os
os.chdir('..')

from thefuzz import process
import cn2an
import logging
import re
from utils import pre_process, pre_process_without_n, standerlize_一般政策性内容

In [4]:

def find_权宜处理_rule_base(docs, 一般政策性内容):
    """
    Return the sentences that match 权宜处理 keywords using regular expression and fuzzy matching.

    Returns:
        list of tuple: The sentences that match 权宜处理 keywords and the matched keywords.
        list of tuple: The index of the matched sentences in the original text and the matched keywords.
    """
    # Ignore the warnings
    logging.getLogger().setLevel(logging.ERROR)
    
    一般政策性内容 = standerlize_一般政策性内容(一般政策性内容)
    
    # Define keywords for regular expression
    keywords = ["结合.*?实际", "根据.*?实际", "因地制宜"]
    pattern = re.compile('|'.join(keywords))
    
    # Define keywords for fuzzy matching
    keywords_fuzzy = ["结合实际", "根据实际", "根据实际情况", "结合实际情况", "结合本地实际", "根据本地实际"]
    
    # Preprocess the document to separate it into paragraphs
    paragraphs = pre_process(docs)
    
    matched_paragraphs = []

    for paragraph in paragraphs:
        clean_paragraph = re.sub(r'\s+', '', paragraph)
        if clean_paragraph in 一般政策性内容:
            continue
        
        match = pattern.search(paragraph)
        if match:
            matched_paragraphs.append((paragraph.strip(), match.group()))
        else:
            best_match = process.extractOne(paragraph, keywords_fuzzy)
            if  best_match[1] >= 60:
                matched_paragraphs.append((paragraph.strip(), best_match[0]))

    # Find the indices of matched paragraphs in the original document
    matched_paragraphs_index = []
    for sentence in matched_paragraphs:
        begin_index = docs.find(sentence[0])
        end_index = begin_index + len(sentence[0])
        matched_paragraphs_index.append((begin_index, end_index))
        
    return matched_paragraphs, matched_paragraphs_index